# 01 — Warehouse population

This notebook is the entry point to the SuSSE warehouse — the shared data store every other tutorial notebook reads from. Three sections, in order:

1. **The model** — what we're building, in one paragraph.
2. **The warehouse** — what's in it, where the data comes from, and the hard constraint it imposes.
3. **Extending the warehouse** — the three ingest patterns and how to add new stations / regions / sources.

Code cells default to **read-only mode** (`RUN_INGEST=False` near the top) — they query the warehouse and show what *would* happen, but don't call external APIs or write rows. Real mutations live in `warehouse/migrations/`. Library architecture and operational practices are documented separately in [`docs/architecture.md`](../../docs/architecture.md).

In [1]:
import shutil
import shlex
import sys
import os

PROJECT_ID = "solar-irradiation-estimation"

if "google.colab" in sys.modules:
    # --- GOOGLE COLAB AUTOMATED SETUP ---
    REPO = "Marconi-Lab/Solar_irradiation"
    BRANCH = "jm/add_model"
    clone_url = f"https://github.com{REPO}.git"
    
    if not os.path.exists("/content/Solar_irradiation/.git"):
        get_ipython().system(f"git clone -q -b {BRANCH} {clone_url} /content/Solar_irradiation")

    get_ipython().run_line_magic('cd', '/content/Solar_irradiation')
    get_ipython().system('pip install -q -e . 2>&1 | tail -3')

    from google.colab import auth
    auth.authenticate_user()
    get_ipython().system(f'gcloud config set project {PROJECT_ID} 2>/dev/null')
    print("Colab setup complete.")

else:
    # --- LOCAL WSL / LINUX AUTOMATED SETUP ---
    print("Running in Local Linux/WSL Environment.")
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    
    # 1. Look for gcloud
    gcloud_path = shutil.which("gcloud")
    
    # Fallback check
    if not gcloud_path:
        home_bin = os.path.expanduser("~/google-cloud-sdk/bin/gcloud")
        if os.path.exists(home_bin):
            gcloud_path = home_bin

    # 2. Prevent installing if already found
    if not gcloud_path:
        print("SDK not found anywhere on the system. Installing now...")
        get_ipython().system('curl -sSL https://sdk.cloud.google.com | bash -s -- --disable-prompts > /dev/null')
        gcloud_path = os.path.expanduser("~/google-cloud-sdk/bin/gcloud")
    else:
        print(f"gcloud detected successfully at: {gcloud_path}")

    # FIXED: Use shlex.quote to safely wrap the windows/WSL path strings for Bash
    safe_gcloud_path = shlex.quote(gcloud_path)

    # 3. Trigger authentication using the safe, quoted path
    print("\nOpening your system browser for Google Cloud verification...")
    get_ipython().system(f'{safe_gcloud_path} auth application-default login')
    
    # 4. Set the project configuration
    get_ipython().system(f'{safe_gcloud_path} config set project {PROJECT_ID} 2>/dev/null')
    print(f"\nLocal setup complete! Active project set to: {PROJECT_ID}")

Running in Local Linux/WSL Environment.
gcloud detected successfully at: /mnt/c/Program Files (x86)/Google/Cloud SDK/google-cloud-sdk/bin/gcloud

Opening your system browser for Google Cloud verification...
Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=2SWyAb03squVUflah2d7C86ffUXPgJ&access_type=offline&code_challenge=2GW5ASawWY_VuiuboT9Ki_BMU6jn1Bs3DtbIyM-He5c&code_challenge_method=S256

gio: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.goo

### Inputs and prerequisites

- **GCP credentials.** Application-default credentials (`gcloud auth application-default login`) for the project listed in `WarehouseConfig` (default: `solar-irradiation-estimation`).
- **A populated warehouse.** The cells below need the table-creation and ingest migrations under `warehouse/migrations/` to have already been applied. The migration index in [`warehouse/migrations/README.md`](../../warehouse/migrations/README.md) records what each script does and whether it has been run.
- **Optional — `CAMS_EMAIL`.** Required only if you flip `RUN_INGEST=True` for any CAMS pattern. Register at [soda-pro.com](https://www.soda-pro.com/) and put the address in `.env`.
- **Optional — `EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD`.** Required for MERRA-2 ingest only; not needed to read the warehouse.

This notebook is **read-only by default** (`RUN_INGEST=False` in the setup cell). Every cell below is safe to run against a populated warehouse without writing or fetching anything.

**Outputs.** No artifacts are written to disk — this notebook builds the reader's mental model of the warehouse. The next acceptance gate (NB 02) consumes the same warehouse to materialise a versioned training-dataset snapshot.

## 1. The model

The SuSSE bias-correction model takes a satellite GHI estimate (from NASA POWER and/or CAMS) plus atmospheric variables at a specific (lat, lon, date), and predicts the **unbiased GHI** for that point. Satellite estimates have systematic 5–15 % biases over Sub-Saharan Africa; the model learns to correct them.

It's trained on **paired data**: at each of our 28 ground-measurement stations, every day where we have both a ground GHI reading and the matching satellite estimate becomes one training example. Once trained, applying the model to a new (lat, lon, date) requires the same shape of input — a satellite estimate plus atmospheric variables for that point. That's what the warehouse is for.

## 2. The warehouse

Both training and inference need the same data shape — `(satellite GHI estimate, atmospheric features) at (lat, lon, date)` — so we pre-stage everything in one BigQuery dataset (`solar-irradiation-estimation.solar_warehouse`) and read from there. For training that's the 28 ground stations × their date ranges; for inference it's any (lat, lon, date) a downstream caller might ask about.

**The hard constraint:** the model can only score (lat, lon, date) tuples for which the warehouse already holds the satellite estimate and atmospheric variables. Adding a new station to the training corpus, or a new region to inference coverage, requires an ingest run first. Section 3 shows how.

The cells below tour the warehouse as it stands today, then summarise where each table's data comes from.

### Setup

In [ ]:
from datetime import date
from pathlib import Path
import logging
import os

import pandas as pd

from susse.warehouse_ops.io import (
    BigQueryClient,
    TableRefs,
    TableSchemas,
    WarehouseConfig,
)
from susse.warehouse_ops.population import (
    BoundingBox,
    CamsSatelliteJob,
    CurationOptions,
    DateRange,
    GridPlan,
    GridSpec,
    GroundFilePlan,
    GroundIngestJob,
    LocationSpec,
    NamedLocationsPlan,
    NasaPowerSatelliteJob,
    Source,
    StandardCsvAdapter,
    VariableCatalog,
    curate_ground,
    populate_dim_variable,
    variables_to_dataframe,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

In [ ]:
# Master toggle: when False, the notebook reads the warehouse and shows what
# would happen, but never calls a satellite API or writes a row. Flip to True
# only when you actually want to ingest.
RUN_INGEST = False

config = WarehouseConfig()  # defaults to solar-irradiation-estimation / solar_warehouse
tables = TableRefs(config=config)
bq = BigQueryClient(config=config)

print(f"project : {config.project_id}")
print(f"dataset : {config.dataset}")
print(f"RUN_INGEST = {RUN_INGEST}")

### Live warehouse tour

What's in the warehouse right now, queried directly. Re-running this notebook over time will show the warehouse growing as we run the ingest jobs.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from susse.warehouse_ops.io import BigQueryClient, TableRefs
bq = BigQueryClient(); refs = TableRefs()
df = bq.query(f'''
        SELECT geohash5, variable_id, COUNT(*) AS n_days
        FROM {refs.merra_daily_vars_long}
        GROUP BY geohash5, variable_id
        ORDER BY geohash5, variable_id
    ''')
print(df.to_string(index=False))

In [ ]:
# Each table or view in the warehouse, with its size and kind. Joining
# __TABLES__ (which has row_count + size_bytes) to INFORMATION_SCHEMA.TABLES
# (which has table_type) so views are labelled as such — __TABLES__ alone
# reports row_count=0 for views (they're computed on demand, not stored),
# which would otherwise look like the views are empty.
tables_df = bq.query(f"""
SELECT t.table_id,
       ts.table_type,
       t.row_count,
       ROUND(t.size_bytes / 1024 / 1024, 1) AS size_mb
FROM `{config.project_id}.{config.dataset}.__TABLES__` AS t
LEFT JOIN `{config.project_id}.{config.dataset}.INFORMATION_SCHEMA.TABLES` AS ts
  ON t.table_id = ts.table_name
ORDER BY t.size_bytes DESC, t.table_id
""")
tables_df

### What's in each table

Each table answers one question; the next cell shows three sample rows from each so you can see actual column names and value ranges before writing queries.

- **Observations — wide format:**
  - `irradiance_daily` — satellite-estimated GHI / DHI / DNI from NASA POWER and CAMS. One row per `(date, geohash5, source)`. Wide layout because the three fluxes are always queried together.
  - `ground_measurements` — curated ground-truth daily GHI from the 28 stations. One row per `(date, location)`. The model's training target. `ground_measurements_raw` is the parallel audit-trail table with the un-curated parsed CSV rows.
- **Observations — long format:**
  - `nasa_daily_vars_long`, `cams_daily_vars_long`, `merra_daily_vars_long` — every auxiliary variable (everything that is not GHI / DHI / DNI). One row per `(date, geohash5, variable_id)` so adding a new variable is a no-op on the schema.
  - `modis_observations` — MODIS observations get their own table (not a `_long` companion) because MODIS products have different cadences (daily / 8-day / 16-day) and are multi-band. One row per `(date, geohash5, product_id, band_id)`. `modis_observations_staging` is the ingest buffer.
- **Catalog:**
  - `dim_variable` — one row per `(variable_id, source)`, with `display_name`, `unit`, `description`, `valid_min` / `valid_max`, and `spatial_resolution_km`. This is the table the variable-catalog cell further down enumerates.
- **Views (no storage cost):**
  - `v_irr_weekly_by_point`, `v_irr_monthly_by_point` — server-side weekly / monthly rollups of `irradiance_daily`. They show `row_count = 0` in the catalog table above because views are computed on demand, not persisted.

In [ ]:
# First three rows of each base table — actual columns + value ranges.
# `tables` is a TableRefs object whose attributes return fully-qualified
# BigQuery table names; iterating gives an idiomatic walk-through.
from IPython.display import display, Markdown

for tbl_attr in (
    "irradiance_daily",
    "ground_measurements",
    "ground_measurements_raw",
    "nasa_daily_vars_long",
    "cams_daily_vars_long",
    "merra_daily_vars_long",
    "modis_observations",
    "dim_variable",
):
    fqn = getattr(tables, tbl_attr)
    display(Markdown(f"**`{tbl_attr}`** — first 3 rows"))
    display(bq.query(f"SELECT * FROM `{fqn}` LIMIT 3"))

In [ ]:
irradiance_summary = bq.query(f"""
SELECT source,
       COUNT(*)                            AS n_rows,
       COUNT(DISTINCT geohash5)            AS n_points,
       MIN(date)                           AS first_date,
       MAX(date)                           AS last_date,
       ROUND(MIN(latitude), 2)             AS min_lat,
       ROUND(MAX(latitude), 2)             AS max_lat,
       ROUND(MIN(longitude), 2)            AS min_lon,
       ROUND(MAX(longitude), 2)            AS max_lon
FROM `{tables.irradiance_daily}`
GROUP BY source
ORDER BY source
""")
irradiance_summary

In [ ]:
ground_summary = bq.query(f"""
SELECT location,
       COUNT(*)                AS n_days,
       MIN(date)               AS first_date,
       MAX(date)               AS last_date,
       ROUND(AVG(lat), 4)      AS lat,
       ROUND(AVG(lon), 4)      AS lon,
       ROUND(AVG(ghi_kwh_m2_day), 2) AS ghi_mean_kwh
FROM `{tables.ground_measurements}`
GROUP BY location
ORDER BY n_days DESC
""")
ground_summary

### Ingest-completeness audit

"Coverage" here means **completeness of the warehouse for the dates / locations / variables we care about** — *not* spatial-area coverage. Two queries are enough to spot-check it before downstream work:

- **Per-variable row counts in the long tables** — answers *"is every variable that's supposed to be ingested actually in the warehouse?"* If a variable is registered in `dim_variable` but missing from its long table, the source's ingest job hasn't been run for it yet.
- **Training-pair availability per ground station** — answers *"for each ground station, on how many of its days do we have a matching satellite estimate at the same `(date, geohash5)`?"* Zero pairs means no training signal for that station, regardless of how much ground or satellite data exists in isolation.

The pair-availability cell is the canary: every station should have `n_nasa_pairs > 0` and `n_cams_pairs > 0` after the 28-station ingest runs. If any are zero, rerun `warehouse/migrations/2026-05-08_a6_ingest_28_ground_stations.py` (it's idempotent — already-cached slices are skipped).

Anything deeper — row-count diffs against a previous run, per-variable date-range gaps, geohash inventories — is one-off enough that we don't keep a dedicated audit script in the repo. Reach for BigQuery directly, or write a fresh migration if the check is worth running again.

In [ ]:
# Per-source variable coverage in the long tables. After Phase A this
# should show ~30+ NASA POWER variables and 6 CAMS aux variables for
# the Uganda 2024 grid.
coverage_df = bq.query(f"""
SELECT 'NASA' AS source, variable_id, COUNT(*) AS n_rows,
       COUNT(DISTINCT geohash5) AS n_points,
       MIN(date) AS min_date, MAX(date) AS max_date
FROM `{tables.nasa_daily_vars_long}`
GROUP BY variable_id
UNION ALL
SELECT 'CAMS' AS source, variable_id, COUNT(*) AS n_rows,
       COUNT(DISTINCT geohash5) AS n_points,
       MIN(date) AS min_date, MAX(date) AS max_date
FROM `{tables.cams_daily_vars_long}`
GROUP BY variable_id
ORDER BY source, variable_id
""")
coverage_df

In [ ]:
# Training-pair availability — for each ground station, how many days have
# a matching satellite estimate at the same (date, geohash5)? Zero pairs
# means no training signal for that station, regardless of how much
# ground or satellite data exists in isolation.
pair_df = bq.query(f"""
WITH g AS (
  SELECT location, date, geohash5
  FROM `{tables.ground_measurements}`
)
SELECT
  g.location,
  COUNT(*) AS n_ground_days,
  COUNTIF(s_nasa.geohash5 IS NOT NULL) AS n_nasa_pairs,
  COUNTIF(s_cams.geohash5 IS NOT NULL) AS n_cams_pairs
FROM g
LEFT JOIN `{tables.irradiance_daily}` s_nasa
  ON g.date = s_nasa.date AND g.geohash5 = s_nasa.geohash5 AND s_nasa.source = 'NASA'
LEFT JOIN `{tables.irradiance_daily}` s_cams
  ON g.date = s_cams.date AND g.geohash5 = s_cams.geohash5 AND s_cams.source = 'CAMS'
GROUP BY g.location
ORDER BY g.location
""")

# Hard assertion: any zero-pair station means the warehouse isn't ready
# for training at that location. Comment out if you're knowingly running
# before A6 has populated satellite estimates for the ground stations.
zero_pair = pair_df[(pair_df["n_nasa_pairs"] == 0) | (pair_df["n_cams_pairs"] == 0)]
if not zero_pair.empty:
    print(f"WARNING: {len(zero_pair)} stations have zero pairs for at least one source.")
    print("Run warehouse/migrations/2026-05-08_a6_ingest_28_ground_stations.py to fix.")
pair_df

### Where the data comes from

Five data sources contribute to the warehouse. Two roles each row plays — *target* (what the model learns to predict) or *feature* (what it consumes as input):

| Source | Role | What it gives | Resolution / cadence | Status |
|---|---|---|---|---|
| **Ground stations** | Target | Daily GHI measured at 28 sites across 7 SSA countries (CrossBoundary, Makerere, Ugandan Min. of Energy) | Daily, point | ✓ ingested via `StandardCsvAdapter` |
| **NASA POWER** | Feature | GHI / DHI / DNI **plus** ~30 atmospheric variables (temp, humidity, AOD, cloud, etc.) | Daily, 0.5° (~55 km) | ✓ ingested |
| **CAMS Radiation** | Feature | All-sky GHI / DHI / DNI / BHI **plus** clear-sky decomposition (McClear) | Daily, 0.05° (~5.5 km) | ✓ ingested |
| **MERRA-2** | Feature | 4 variables: AOD extinction / scattering / analysis at 550 nm + total precipitable water | Hourly native, daily-aggregated | ✓ in flight via migration B8 (still backfilling stations) |
| **MODIS** | Feature | 3 surface-state products: nadir BRDF reflectance band 1 (red, daily 500 m); day land-surface temperature (8-day, 1 km); NDVI (16-day, 250 m) | Mixed (daily / 8-day / 16-day) | ✓ in flight via migration C8 |

The variables actually pulled per source are listed in `VariableCatalog` (see the catalog cell below); each `VariableSpec`'s `description` field explains why it's included. We deliberately ingest a subset of what each source offers — see the catalog for details.

Two warehouse-layout notes:
- **Irradiance estimates** (GHI/DHI/DNI) from NASA POWER and CAMS land in a wide table `irradiance_daily`, distinguished by the `source` column.
- **Auxiliary variables** land in long-format companion tables, one per source: `nasa_daily_vars_long`, `cams_daily_vars_long`, `merra_daily_vars_long`. Same schema, easy to UNION when assembling features.
- **MODIS** has its own `modis_observations` table (not a `_long` companion) because its products run at different cadences (daily / 8-day / 16-day) and several are multi-band.
- **Ground truth** lives in `ground_measurements` (curated) and `ground_measurements_raw` (audit trail of the parsed CSVs).
- **`dim_variable`** is the canonical catalog: one row per `(variable_id, source)` with units and metadata.

In [ ]:
# Browse the variable catalog. Whenever you write a FeatureSelection
# (NB 02) or a FeatureSpec (NB 03), every variable_id you reference must
# appear below — so this is the cell to consult when picking inputs.
#
# Each row carries a `description` field explaining what the variable
# measures; the markdown cell that follows groups them by physical role.
print("Source enum members:", [s.value for s in Source])

catalog_df = variables_to_dataframe(VariableCatalog.all_variables())
print(f"Catalog: {len(catalog_df)} entries across {catalog_df['source'].nunique()} sources.\n")

# Show description unwrapped so the per-variable rationale is visible.
pd.set_option("display.max_colwidth", 120)
catalog_df[["source", "variable_id", "display_name", "unit", "description"]]

### Why these variables (and not others)

The catalog above is grouped by source for the table view, but the model cares about *what each variable physically represents*. Reading the rows along that axis, five categories emerge.

**1. Radiation budget — the target and its near-relatives.** `ghi` / `dhi` / `dni` (NASA + CAMS) are the all-sky satellite estimates the model is asked to bias-correct; ground GHI is the target. CAMS additionally publishes the **clear-sky** decomposition (`ghi_clear`, `bhi_clear`, `dhi_clear`, `dni_clear`) from the McClear model, plus extraterrestrial GHI (`ghi_extra`). Dividing all-sky by clear-sky yields the **clear-sky index** `kt = ghi / ghi_clear`, a normalised cloudiness signal that strips out latitude / season / orbit effects — NB 03 uses this as a derived feature. NASA POWER pre-computes its own `clearness_index`, so we pass it through as a feature instead of re-deriving.

**2. Atmospheric attenuation — what makes the all-sky deviate from clear-sky.** Four physical drivers are kept:

- **Aerosols** — `aod_550`, `aod_550_adj`, `aod_840` (NASA POWER); `aod_550_extinction`, `aod_550_scattering`, `aod_550_analysis` (MERRA-2). AOD captures Saharan-dust outbreaks and biomass-burning plumes that satellite retrievals systematically under-correct over Sub-Saharan Africa. The 840 nm band is more sensitive to coarse-mode dust; MERRA-2's extinction / scattering split lets the model distinguish absorbing aerosols (smoke) from purely scattering ones (sulfate haze).
- **Water vapour** — `precipitable_water` (NASA POWER and MERRA-2). Major near-infrared absorber and a proxy for cloudiness regime.
- **Clouds** — `cloud_amount`, `cloud_amount_dat`, `cloud_visible_optical_depth`. The single biggest source of GHI variance — the dominant lever the model has.
- **Ozone** — `total_column_ozone`. UV absorber; small but non-negligible effect on the shortwave budget.

**3. Surface state — what the radiation interacts with at the bottom.** Three flavours of NASA POWER albedo (`surface_albedo`, `all_sky_surface_albedo`, `clear_sky_albedo`) cover bulk reflectivity. MODIS adds higher-resolution surface descriptors: nadir red-band reflectance (`MCD43A4_Nadir_Reflectance_Band1`, 500 m), daytime land-surface temperature (`MOD11A2_LST_Day_1km`, 1 km), and NDVI vegetation index (`MOD13Q1_250m_16_days_NDVI`, 250 m). These are slow-changing land-cover proxies that distinguish, e.g., desert from cropland at scales finer than POWER's 0.5° grid.

**4. Boundary-layer meteorology — secondary modulators.** Standard NASA POWER met fields: `temperature`, `temperature_range`, `specific_humidity`, `relative_humidity`, `surface_pressure`, `wind_speed`, `northern_wind`, `surface_air_density`, `airmass`, `planetary_boundary`, `surface_roughness`, `zero_plane_displacement`. They influence the model's ability to disambiguate clear-but-hazy from cloudy days, especially in the dry season.

**5. Hydrology — physically informative even if rarely the dominant signal.** `precipitation_corrected`, `evaporation_land`, `evapotranspiration_energy`, `surface_soil_wetness`, `longwave_downward_irr`. Useful for distinguishing convective-rainy from dry-clear regimes; also potential targets if the project ever expands beyond GHI.

**What we deliberately don't ingest.** NASA POWER offers ~150 daily variables and CAMS ~30 — we keep only the subset above. Variables omitted because they don't carry signal for *GHI* bias correction in *Sub-Saharan Africa*: snow depth (negligible coverage), agricultural-yield indices (downstream not upstream), sub-daily wind components (we work at daily cadence), soil-temperature profiles (not directly radiative). Adding any of them is a one-line edit to `VariableCatalog`; the warehouse just hasn't fetched data we won't use.

The per-variable rationale is in each `VariableSpec.description` (the rightmost column of the catalog above). When in doubt about why a particular variable is on the list, find its row.

### Quick example: training vs inference

The cleanest way to see what the warehouse delivers to the model is to load it from both sides:

- **Training** — at a ground station, we get the *target* (ground GHI) plus the *features* (NASA + CAMS satellite GHI estimates and atmospheric variables). One call to `FeatureService.build_training_pairs(...)` assembles the join.
- **Inference** — at any (lat, lon, date) the warehouse covers, we get the same features but no target (that's what the model is going to predict). One call to `FeatureService.build_inference_features(...)` returns a single-row frame.

The cells below load **kampala for 2024** (training) and a **central-Uganda grid point** at `(0.5179, 32.4715)` for the same year (inference), then plot them side by side. The training column shows the ground-truth line the model is learning to match; the inference column shows what's available for prediction at points where we have *no* ground measurement — which is the whole reason for the warehouse's inference cache.

In [ ]:
# One call to FeatureService.build_training_pairs assembles the full
# training shape: target (ground GHI), primary inputs (NASA + CAMS GHI),
# and any selected NASA atmospheric features — all aligned on (date,
# geohash5). For inference at a new (lat, lon, date) you'd call
# build_inference_features with the same set of variables.
from susse.datasets import FeatureSelection, FeatureService

fs = FeatureService(bq=bq)  # tables defaults to TableRefs(config=bq.config)
selection = FeatureSelection(
    nasa_variable_ids=("aod_550", "precipitable_water", "cloud_amount"),
)
example = fs.build_training_pairs(
    selection=selection,
    date_start=date(2024, 1, 1),
    date_end=date(2024, 11, 25),
    locations=("kampala",),
)

# Date-indexed view for the resampling-based plots below. Defining it
# once here keeps the plot cells order-independent.
timeseries = (
    example.assign(date=pd.to_datetime(example["date"]))
    .set_index("date")
    .sort_index()
)

print(f"Loaded {len(example)} daily rows for kampala (2024).")
print("Columns:", list(example.columns))
example.head(3)

In [ ]:
# Inference-side feature lookup for a single (lat, lon, date). This is
# the call shape the portal would make in production: given a request
# location and date, return the satellite GHI estimates and the model's
# atmospheric features. Ground GHI is NOT in the result — that's what
# the model will predict.
inference_lat, inference_lon = 0.5179, 32.4715  # central Uganda grid point
inference_today = fs.build_inference_features(
    selection=selection,
    target_date=date(2024, 6, 15),
    lat=inference_lat,
    lon=inference_lon,
)
print("build_inference_features returns one row per (lat, lon, date):")
inference_today

In [ ]:
# For the year-long plot we need 365 inference rows. Calling
# build_inference_features in a loop would issue 365 BQ queries; instead
# we fetch the same shape in one query against the underlying tables.
# This is functionally what build_inference_features does internally,
# just vectorised across dates for one location.
import pygeohash

inference_geohash = pygeohash.encode(inference_lat, inference_lon, precision=5)
inference_year = bq.query(f"""
WITH irr AS (
  SELECT date,
         MAX(IF(source='NASA', ghi_kwh_m2_day, NULL)) AS sat_ghi_nasa_kwh_m2_day,
         MAX(IF(source='CAMS', ghi_kwh_m2_day, NULL)) AS sat_ghi_cams_kwh_m2_day
  FROM `{tables.irradiance_daily}`
  WHERE date BETWEEN DATE('2024-01-01') AND DATE('2024-11-25')
    AND geohash5 = '{inference_geohash}'
  GROUP BY date
),
nasa AS (
  SELECT * FROM (
    SELECT date, variable_id, value
    FROM `{tables.nasa_daily_vars_long}`
    WHERE date BETWEEN DATE('2024-01-01') AND DATE('2024-11-25')
      AND geohash5 = '{inference_geohash}'
      AND variable_id IN ('aod_550', 'precipitable_water', 'cloud_amount')
  )
  PIVOT (ANY_VALUE(value) FOR variable_id IN (
      'aod_550' AS nasa_aod_550,
      'precipitable_water' AS nasa_precipitable_water,
      'cloud_amount' AS nasa_cloud_amount
  ))
)
SELECT * FROM irr LEFT JOIN nasa USING (date)
ORDER BY date
""")

inference_ts = (
    inference_year.assign(date=pd.to_datetime(inference_year["date"]))
    .set_index("date")
    .sort_index()
)
print(f"Loaded {len(inference_year)} inference rows for ({inference_lat}, {inference_lon}).")
inference_year.head(3)

In [ ]:
# Two-column GHI plot — training (with ground truth) vs inference (no
# ground truth). Weekly-averaged so the seasonal pattern is visible
# under the daily noise.
import matplotlib.pyplot as plt

weekly_train_ghi = (
    timeseries[
        ["y_ghi_kwh_m2_day", "sat_ghi_nasa_kwh_m2_day", "sat_ghi_cams_kwh_m2_day"]
    ]
    .resample("W")
    .mean()
)
weekly_inf_ghi = (
    inference_ts[["sat_ghi_nasa_kwh_m2_day", "sat_ghi_cams_kwh_m2_day"]]
    .resample("W")
    .mean()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

# Left: training (kampala, 2024) — model sees ground GHI as the target.
weekly_train_ghi["sat_ghi_nasa_kwh_m2_day"].plot(ax=axes[0], label="NASA POWER", color="C0")
weekly_train_ghi["sat_ghi_cams_kwh_m2_day"].plot(ax=axes[0], label="CAMS", color="C1")
weekly_train_ghi["y_ghi_kwh_m2_day"].plot(ax=axes[0], label="Ground truth", color="black", linewidth=2)
axes[0].set_title(f"Training: {example['location'].iloc[0]} (Uganda)")
axes[0].set_ylabel("GHI (kWh/m²/day, weekly mean)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Right: inference (Uganda grid point, 2024) — same features but no ground truth.
weekly_inf_ghi["sat_ghi_nasa_kwh_m2_day"].plot(ax=axes[1], label="NASA POWER", color="C0")
weekly_inf_ghi["sat_ghi_cams_kwh_m2_day"].plot(ax=axes[1], label="CAMS", color="C1")
axes[1].set_title(f"Inference: ({inference_lat}, {inference_lon}) (central Uganda)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.suptitle("Weekly mean GHI — training has the target line; inference does not", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Two-column atmospheric features — same three NASA POWER variables at
# both locations. These go into the model alongside the satellite GHI
# estimate. Inference uses identical feature shape to training; only
# the ground-truth target is missing on the inference side.
weekly_train_vars = (
    timeseries[["nasa_aod_550", "nasa_precipitable_water", "nasa_cloud_amount"]]
    .resample("W")
    .mean()
)
weekly_inf_vars = (
    inference_ts[["nasa_aod_550", "nasa_precipitable_water", "nasa_cloud_amount"]]
    .resample("W")
    .mean()
)

fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True, sharey="row")

# Left column: training location.
weekly_train_vars["nasa_aod_550"].plot(ax=axes[0, 0], color="C2")
axes[0, 0].set_title(f"Training: {example['location'].iloc[0]}")
axes[0, 0].set_ylabel("AOD @ 550 nm")
weekly_train_vars["nasa_precipitable_water"].plot(ax=axes[1, 0], color="C3")
axes[1, 0].set_ylabel("Precipitable water (cm)")
weekly_train_vars["nasa_cloud_amount"].plot(ax=axes[2, 0], color="C4")
axes[2, 0].set_ylabel("Cloud amount (%)")
axes[2, 0].set_xlabel("Week")

# Right column: inference location.
weekly_inf_vars["nasa_aod_550"].plot(ax=axes[0, 1], color="C2")
axes[0, 1].set_title(f"Inference: ({inference_lat}, {inference_lon})")
weekly_inf_vars["nasa_precipitable_water"].plot(ax=axes[1, 1], color="C3")
weekly_inf_vars["nasa_cloud_amount"].plot(ax=axes[2, 1], color="C4")
axes[2, 1].set_xlabel("Week")

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle("NASA POWER atmospheric features — weekly mean", y=1.00)
plt.tight_layout()
plt.show()

## 3. Extending the warehouse

Whenever you want to train on a new station, score a new region, or pull a new variable, you ingest first. There are exactly three patterns, distinguished by what drives the location list:

| Pattern | Plan type | Use when |
|---|---|---|
| 1 | `NamedLocationsPlan` | You have specific points in mind — typically the 28 ground stations whose history you want covered for training. |
| 2 | `GridPlan` | You want uniform geographic coverage of a region — typically Uganda, or eventually all of sub-Saharan Africa, for the inference cache. |
| 3 | `GroundFilePlan` | A new ground-truth dataset has arrived from a station owner, partner, or publication. |

All three share the same lifecycle: **fetch → validate → server-side MERGE**. They differ only in how they resolve the location list and which fetcher they call. Two non-negotiable properties hold across every pattern:

- **Idempotency** — the coverage check pre-queries BigQuery for existing keys (date, geohash5, variable, source) and skips API calls for slices already loaded. Re-running a job is cheap. Ctrl+C mid-run is safe.
- **Migrations vs notebook cells** — the demo cells below run with `RUN_INGEST=False` by default and exist as reference. Real multi-station ingest belongs in `warehouse/migrations/*.py` (idempotent, dated, versioned in git). Don't flip `RUN_INGEST=True` and rerun the whole notebook; write a migration. See [`docs/architecture.md`](../../docs/architecture.md) for the full operational guide.

### Pattern 1 — Named-location satellite ingest

Fetch a satellite source for a discrete set of locations — typically the ground stations whose history we want covered for training. Below: a plan for **Uganda 2025** at ground-station coordinates. The job's coverage check skips slices already cached, so re-running is cheap.

In [ ]:
# Pull station coordinates from the warehouse rather than hardcoding them.
stations = bq.query(f"""
SELECT location, ROUND(AVG(lat), 6) AS lat, ROUND(AVG(lon), 6) AS lon
FROM `{tables.ground_measurements}`
WHERE location IN ('kampala','lira','tororo','soroti','wadelai')
GROUP BY location
ORDER BY location
""")
stations

In [ ]:
uganda_locations = tuple(
    LocationSpec(name=row.location, lat=float(row.lat), lon=float(row.lon))
    for row in stations.itertuples(index=False)
    if row.location in {"kampala", "lira", "tororo", "soroti", "wadelai"}
)

uganda_2025 = DateRange(start=date(2025, 1, 1), end=date(2025, 1, 31))

# Fetch the full set of NASA POWER variables this project tracks.
nasa_vars = VariableCatalog.for_source(Source.NASA_POWER)
cams_vars = VariableCatalog.for_source(Source.CAMS)

uganda_nasa_plan = NamedLocationsPlan(
    source=Source.NASA_POWER,
    date_range=uganda_2025,
    locations=uganda_locations,
    variables=nasa_vars,
)
uganda_cams_plan = NamedLocationsPlan(
    source=Source.CAMS,
    date_range=uganda_2025,
    locations=uganda_locations,
    variables=cams_vars,
)

for plan in (uganda_nasa_plan, uganda_cams_plan):
    print(plan.describe())

In [ ]:
if RUN_INGEST:
    nasa_job = NasaPowerSatelliteJob(bq=bq, refs=tables)
    for plan in (uganda_nasa_plan,):
        result = nasa_job.run(plan)
        print(result.summary())
else:
    print("RUN_INGEST=False — skipping NASA POWER named-location runs.")
    print("Each plan above describes one job invocation; flip RUN_INGEST to execute.")

In [ ]:
# CAMS requires a registered email. Set CAMS_EMAIL in your environment
# (or .env) before running this cell.
if RUN_INGEST and os.getenv("CAMS_EMAIL"):
    cams_job = CamsSatelliteJob(bq=bq, refs=tables)
    for plan in (uganda_cams_plan,):
        result = cams_job.run(plan)
        print(result.summary())
elif RUN_INGEST:
    print("RUN_INGEST=True but CAMS_EMAIL is not set — skipping CAMS runs.")
    print("Register an email at https://www.soda-pro.com/ and put it in .env as CAMS_EMAIL.")
else:
    print("RUN_INGEST=False — skipping CAMS named-location runs.")

### Pattern 2 — Grid satellite ingest

The portal needs a satellite GHI estimate for any (lat, lon) in the region a user might query. Pre-cache them with a `GridPlan` over a bounding box at the source's native resolution, so the portal can serve cached estimates directly. Below: a tiny demo grid over a 0.5° × 0.5° patch of central Uganda at CAMS native 0.05° (121 grid points). Real region-wide grids run as separate, deliberate batches via migrations.

In [ ]:
uganda_demo_bbox = BoundingBox(
    min_lat=0.10, max_lat=0.60,
    min_lon=32.30, max_lon=32.80,
)
demo_grid = GridSpec(bbox=uganda_demo_bbox, lat_step=0.05, lon_step=0.05)
demo_dates = DateRange(start=date(2025, 1, 1), end=date(2025, 1, 7))

demo_grid_plan = GridPlan(
    source=Source.CAMS,
    date_range=demo_dates,
    grid=demo_grid,
    variables=cams_vars,
)
print(demo_grid_plan.describe())
print(f"Grid points: {demo_grid.n_points}")

In [ ]:
if RUN_INGEST and os.getenv("CAMS_EMAIL"):
    cams_job = CamsSatelliteJob(bq=bq, refs=tables)
    grid_result = cams_job.run(demo_grid_plan)
    print(grid_result.summary())
else:
    print("Grid ingest skipped (RUN_INGEST=False or CAMS_EMAIL not set).")
    print("This plan would call CAMS once per grid point that isn't already cached.")

### Pattern 3 — Ground-truth ingest

Future ground-truth additions arrive in heterogeneous formats from publications, ministries, and partner organisations. A `GroundSourceAdapter` parses one provider's CSV layout; downstream the curation function and MERGE loader are unchanged.

The Makerere and Min.-of-Energy CSVs (and any further providers added under `data/ground_measurements/`) all share `(datetime, ghi, location, latitude, longitude)`, so a single `StandardCsvAdapter` handles them. New layout = new adapter subclass.

The demo runs end-to-end against the real Min.-of-Energy Soroti file. Because every row is already in `ground_measurements_raw`, the idempotent coverage check causes the job to add zero rows — a real safety check that the pipeline doesn't double-write.

In [ ]:
# Real on-disk file from the project's data directory.
soroti_csv = Path("../../data/ground_measurements/ministry_energy_ug/soroti.csv").resolve()
assert soroti_csv.exists(), f"Expected {soroti_csv} to exist."

adapter = StandardCsvAdapter(source_id="min_energy_ug")
raw_df = adapter.parse(soroti_csv)
print(f"Parsed {len(raw_df)} raw rows.")
raw_df.head()

In [ ]:
# QC levels currently in use. `qc_level` is a free-form string stamped
# on every curated row by `CurationOptions(qc_level=...)`. Downstream
# (NB 02's FeatureSelection) lets you keep only the tags you trust —
# e.g. `qc_levels=("pass",)` to exclude `auto` tags from training.
qc_levels_df = bq.query(f"""
SELECT qc_level, COUNT(*) AS n_rows, COUNT(DISTINCT location) AS n_stations
FROM `{tables.ground_measurements}`
GROUP BY qc_level
ORDER BY n_rows DESC
""")
qc_levels_df

In [ ]:
# Curation is a pure transformation — no I/O. Run it on the parsed raw rows
# to see what would land in `ground_measurements`.
curated_df = curate_ground(
    raw_df,
    CurationOptions(qc_level="auto", version="v1", geohash_precision=5),
)
print("Curated columns:", list(curated_df.columns))
curated_df.head()

In [ ]:
# Run the full ingest path. Idempotent: if the data is already in the warehouse
# (which it is, for this CSV), the job adds zero rows and reports them all as
# already cached.
if RUN_INGEST:
    ground_plan = GroundFilePlan(
        source_id="CBE",
        file_path=somalia_csv,
        adapter_id=adapter.adapter_id,
    )
    ground_job = GroundIngestJob(bq=bq, adapter=adapter, refs=tables)
    ground_result = ground_job.run(ground_plan)
    print(ground_result.summary())
else:
    print("Ground ingest skipped (RUN_INGEST=False).")

### Extension recipes

| To add… | Do this |
|---|---|
| **A new ground station** | Drop the CSV at `data/ground_measurements/<provider>/<station>.csv`, pick or write a `GroundSourceAdapter`, run `GroundIngestJob` (Pattern 3), then re-run `2026-05-08_a6_ingest_28_ground_stations.py` — it pulls station metadata directly from `ground_measurements` and the coverage check skips already-ingested ones. |
| **A new ground-truth source with a different CSV layout** | Subclass `GroundSourceAdapter` and implement `parse(file_path) -> DataFrame` returning the canonical `(datetime, ghi, location, latitude, longitude)` schema. Everything downstream is unchanged. The adapter is the only thing that knows about the partner's column layout. |
| **A new satellite source** | Add a fetcher under `susse.api_clients/<source>/`, a `Source` enum value, `VariableSpec`s in `VariableCatalog`, a `TableSchema` + DDL + `CREATE TABLE` migration, a `BaseSatelliteJob` subclass, and an ingest migration. The MERRA-2 wire-up (Phase B, files prefixed `b1`–`b8`) is the worked template. |
| **A new region (inference coverage)** | Build a `GridPlan` with a `BoundingBox` for the region and run an existing satellite job (Pattern 2 below). Cost scales linearly with `n_grid_points × n_days × n_sources`; CAMS at native 0.05° is the expensive one. Densify pragmatically — start coarser, densify on demand. |
| **A new variable from an existing source** | Add a `VariableSpec` to the relevant `VariableCatalog` block, run the dim_variable migration to register it, then re-run the source's ingest. The job uses the catalog as its source-of-truth for what to fetch. |

The pattern demos that follow are read-only by default — they describe what *would* happen.

## What's next

This is the first of five tutorial notebooks that walk through the SuSSE pipeline end-to-end:

- **Notebook 02** assembles a versioned **TrainingDataset** snapshot from the warehouse — a parquet artifact tying a specific slice of the warehouse to a training run.
- **Notebooks 03–05** build the model side: preprocessor, model factory, trainer with native W&B integration.
- **Notebooks 06 and 07** (evaluation, inference API) are planned and not yet shipped — see the per-notebook README for the current status.

You're ready for notebook 02 when the pair-availability cell shows non-zero NASA + CAMS pairs at every station, the `dim_variable` catalog is populated, and you've run the MERRA-2 ingest if you want those features in the dataset.

If something here doesn't add up — a coverage warning, a confusing class name, a missing extension path — **edit this notebook**. It's the first thing the next person on this codebase will read.